# Classification de texte : régression logistique multinomiale from scratch

**Série NLP — carnet 11.** Ce carnet construit, entraîne et évalue un classifieur de texte **discriminatif** en numpy pur : softmax, entropie croisée et descente de gradient avec régularisation L2. Aucun framework de descente de gradient n'est utilisé — `scikit-learn` intervient uniquement comme **juge extérieur**, à la fin.

## Où ce carnet se situe dans la série

| Carnet | Objet | Modèle |
|---|---|---|
| 06 | recherche d'information | TF-IDF, BM25 — pondération, pas de classification |
| 07 à 09 | plongements de mots | co-occurrence/PPMI/SVD, word2vec, fastText |
| 10 | classification de texte | Naive Bayes — **génératif** |
| **11** | classification de texte | régression logistique — **discriminatif** |

La différence entre 10 et 11 n'est pas un détail d'implémentation.

- Naive Bayes **modélise** `P(mots | classe)` et suppose les mots **indépendants** conditionnellement à la classe. L'hypothèse est fausse sur du texte réel, et elle est fausse d'une manière qui **compte** : des mots corrélés sont comptés plusieurs fois.
- La régression logistique modélise **directement** `P(classe | mots)` et ne suppose aucune indépendance. Elle apprend des **poids** qui se compensent entre eux.

Le prix de cette différence est une optimisation itérative au lieu d'un simple comptage. C'est ce prix que ce carnet rend visible : une courbe de convergence, un balayage de régularisation, et une comparaison de bout en bout avec l'implémentation de référence.

## Objectifs : ce que vous saurez faire à la fin

1. écrire la fonction softmax et montrer pourquoi on retire le maximum avant l'exponentielle ;
2. dériver le gradient de l'entropie croisée softmax et vérifier qu'il tient en `P - Y` ;
3. entraîner un modèle par descente de gradient pleine passe et lire sa convergence ;
4. mesurer une précision **hors échantillon** et la comparer à une baseline ;
5. expliquer ce qu'un balayage de `lambda` révèle, et où se situe le compromis.

## Plan

1. Pourquoi le corpus jouet ne suffit pas
2. Représentation : sac de mots, fréquences relatives
3. Le modèle : softmax, entropie croisée, gradient L2 — et la précision hors échantillon mesurée
4. Cross-check SOTA contre `scikit-learn`
5. Ce que la régularisation fait — et ce qu'elle ne fait pas
6. Synthèse
7. Exercices, puis références

Les titres numérotés du carnet suivent exactement ce plan ; la précision hors échantillon est la dernière lecture de la section 3, elle n'a pas de rubrique à elle.
**Navigation** : [Précédent : Naive Bayes](10-Classement-Texte-NaiveBayes-FromScratch.ipynb) | [Index NLP](README.md) | [Version C#](11-Classement-Texte-RegressionLogistique-FromScratch-CSharp.ipynb)

**Prérequis** : Python, algèbre linéaire et probabilités. **Durée** : 90 minutes.


In [1]:
import re
import numpy as np


def tokeniser(texte):
    """Sac de mots minimal : minuscules, suites de lettres (l'apostrophe separe)."""
    return re.findall(r"[a-z]+", texte.lower())


# --- Corpus jouet du carnet 10 : le socle commun du duo (4 categories x 8 documents) ---
CORPUS = {
    "sport": [
        "l'equipe du stade remporte le match sur un dernier but",
        "l'arbitre siffle la fin du match dans le stade",
        "le coach motive l'equipe avant le match decisif",
        "le gardien arrete le penalty au dernier moment",
        "les joueurs celebrent la victoire dans le stade",
        "le buteur marque le but decisif du match",
        "le score final du match est historique au stade",
        "l'equipe entree au stade pour le match de la saison",
    ],
    "tech": [
        "le developp corrige un bug dans le logiciel",
        "le serveur renvoie une erreur dans le log du systeme",
        "la puce integre un accelerateur dans le processeur",
        "le framework compile le code en quelques secondes",
        "l'application consomme la memoire au demarrage du systeme",
        "l'algorithme accelere le calcul dans le logiciel",
        "l'interface du logiciel interroge le serveur du systeme",
        "le code du systeme compile sans erreur",
    ],
    "science": [
        "l'experience mesure la constante avec une grande precision",
        "le paper publie les resultats de l'observation",
        "l'hypothese est confirmee par les donnees recueillies",
        "le chercheur analyse les echantillons au microscope",
        "la theorie predit le comportement du systeme physique",
        "l'article rapporte une mesure de l'energie",
        "les mesures de l'experience valident le modele theorique",
        "l'observation confirme le resultat de l'experience",
    ],
    "finance": [
        "la bourse enregistre une hausse des indices ce matin",
        "la banque annonce un resultat net en hausse",
        "l'action cote plus haut a la fin de la seance",
        "l'investisseur choisit l'obligation pour le credit",
        "le taux d'interet guide la decision d'emprunt",
        "le portefeuille subit une baisse des valeurs",
        "le dividende est verse aux actionnaires du marche",
        "le marche anticipe une revision des taux",
    ],
}
CATEGORIES = list(CORPUS)

print("corpus jouet :", {c: len(CORPUS[c]) for c in CATEGORIES})
print("total        :", sum(len(v) for v in CORPUS.values()), "documents")
print("exemple      :", CORPUS["sport"][0])

corpus jouet : {'sport': 8, 'tech': 8, 'science': 8, 'finance': 8}
total        : 32 documents
exemple      : l'equipe du stade remporte le match sur un dernier but


## 1. Pourquoi le corpus jouet ne suffit pas

Le carnet 10 travaille sur 32 documents courts, 8 par catégorie. Ce petit corpus offre un banc de test très limité : huit documents seulement étaient réservés au test dans le carnet 10. Une telle mesure ne suffit pas à caractériser la généralisation.

Ce carnet garde ce corpus comme **socle de vocabulaire** — c'est lui qui donne au duo 10/11 la même langue — mais construit à partir de lui un banc où les classes se **chevauchent** :

- le vocabulaire reste **commun** aux quatre catégories, y compris les mots grammaticaux qui n'appartiennent à aucune ;
- chaque catégorie en sur-représente une partie, sans jamais l'isoler ;
- un document est un **mélange** : ses mots sont tirés d'une distribution `lambda * P(categorie) + (1 - lambda) * P(autre categorie)` ;
- `lambda` suit une loi `Beta(5, 2)`, dont la moyenne vaut `alpha / (alpha + beta) = 5/7`, soit `0,714` : un document **penche** vers sa classe, sans jamais l'isoler. Les documents où `lambda < 0.65` sont comptés et affichés — ce sont eux qui rendent le banc difficile, et le carnet les mesure au lieu de les cacher.

Le générateur est **déterministe** (graine fixée) : le banc est reproductible, et les mesures de ce carnet sont comparables d'une exécution à l'autre. Le découpage apprentissage/test est fait **avant** tout entraînement, une seule fois.

In [2]:
# --- Banc mesurable : meme vocabulaire, classes qui se chevauchent, graine fixe ---
vocabulaire = sorted({m for docs in CORPUS.values() for d in docs for m in tokeniser(d)})
position = {m: i for i, m in enumerate(vocabulaire)}


def distribution(docs, alpha=0.05):
    """Distribution de mots d'une categorie, lissee sur tout le vocabulaire.

    alpha evite les probabilites nulles : sans lui, un mot absent d'une categorie ne
    pourrait jamais y apparaitre, et le melange de deux categories serait artificiel.
    """
    compte = np.zeros(len(vocabulaire))
    for d in docs:
        for m in tokeniser(d):
            compte[position[m]] += 1.0
    total = compte.sum()
    return (compte + alpha) / (total + alpha * len(vocabulaire))


P_CLASSE = {c: distribution(CORPUS[c]) for c in CATEGORIES}

GRAINE = 20261029
rng = np.random.RandomState(GRAINE)
N_PAR_CLASSE = 60

documents, etiquettes, lambdas = [], [], []
for classe in CATEGORIES:
    autres = [a for a in CATEGORIES if a != classe]
    for _ in range(N_PAR_CLASSE):
        melange = autres[rng.randint(len(autres))]
        lam = float(rng.beta(5.0, 2.0))
        probs = lam * P_CLASSE[classe] + (1.0 - lam) * P_CLASSE[melange]
        probs = probs / probs.sum()
        n_mots = int(rng.randint(25, 41))
        indices = rng.choice(len(vocabulaire), size=n_mots, p=probs)
        documents.append(" ".join(vocabulaire[i] for i in indices))
        etiquettes.append(classe)
        lambdas.append(lam)

documents = np.array(documents)
etiquettes = np.array(etiquettes)
lambdas = np.array(lambdas)

# Decoupage fait AVANT tout entrainement, une seule fois.
ordre = np.random.RandomState(GRAINE + 1).permutation(len(documents))
n_train = int(0.6 * len(documents))
idx_train, idx_test = ordre[:n_train], ordre[n_train:]

ambigus = lambdas < 0.65
print("banc genere     :", len(documents), "documents,", len(vocabulaire), "mots de vocabulaire")
print("apprentissage   :", len(idx_train), "| test :", len(idx_test))
print("documents ambigus (lambda < 0.65) :",
      int(ambigus.sum()), "soit", f"{ambigus.mean():.1%}", "du banc")
print("\nexemple de document genere :", documents[0][:90], "...")

banc genere     : 240 documents, 138 mots de vocabulaire
apprentissage   : 144 | test : 96
documents ambigus (lambda < 0.65) : 75 soit 31.2% du banc

exemple de document genere : stade pour match resultat publie victoire de du la le du moment stade est arrete le entree ...


## 2. Représentation : sac de mots, fréquences relatives

Comme au carnet 06, chaque document devient un vecteur sur un vocabulaire fixe. Deux choix méritent d'être explicités :

- **fréquences relatives** plutôt que comptes bruts : un document long ne doit pas peser plus lourd dans la fonction de perte du seul fait de sa longueur ;
- **pas de TF-IDF ici** : le TF-IDF est le sujet du carnet 06. Le but de ce carnet est de faire porter la discrimination au **modèle**, pas à la pondération des entrées.

La baseline est la **classe majoritaire** : prédire toujours la catégorie la plus fréquente du jeu d'apprentissage. C'est le plancher qu'un classifieur doit dépasser pour avoir appris quelque chose. Sur un banc équilibré elle vaut 25 % — un modèle à 25 % n'a rien appris du texte, il a appris le compte des classes.

In [3]:
def construire_matrice(docs, vocabulaire, tokenizer=tokeniser):
    """Fréquences relatives ; tokenizer peut accepter des listes pré-tokenisées."""
    position_locale = {mot: i for i, mot in enumerate(vocabulaire)}
    X = np.zeros((len(docs), len(vocabulaire)))
    for i, d in enumerate(docs):
        for m in tokenizer(d):
            if m in position_locale:  # les mots hors vocabulaire sont ignores
                X[i, position_locale[m]] += 1.0
    longueurs = X.sum(axis=1, keepdims=True)
    longueurs[longueurs == 0] = 1.0
    return X / longueurs


X = construire_matrice(documents, vocabulaire)
X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = etiquettes[idx_train], etiquettes[idx_test]
Y_train = np.eye(len(CATEGORIES))[[CATEGORIES.index(c) for c in y_train]]


def baseline_majoritaire(y_train, y_test):
    """Plancher : predire toujours la classe la plus frequente de l'apprentissage."""
    valeurs, comptes = np.unique(y_train, return_counts=True)
    return float((y_test == valeurs[comptes.argmax()]).mean())


print("matrice X      :", X.shape, "(", X.shape[0], "documents x", X.shape[1], "mots )")
print("densite        :", f"{np.count_nonzero(X) / X.size:.1%}", "des cellules non nulles")
print("somme ligne 0  :", round(float(X[0].sum()), 6), "(frequences relatives)")
print("\nbaseline majoritaire sur le test :", f"{baseline_majoritaire(y_train, y_test):.2%}")
# Un vocabulaire alternatif et un mot inconnu exercent le contrat local.
assert np.allclose(construire_matrice(["b b inconnu", "a"], ["b"]), [[1.0], [0.0]])
assert np.allclose(construire_matrice(["xyz"], ["xyz"]), [[1.0]])
assert np.allclose(construire_matrice([["a", "b", "a_b"]], ["a", "b", "a_b"], tokenizer=lambda tokens: tokens), [[1/3, 1/3, 1/3]])
print("vocabulaire alternatif, OOV et bigramme pré-tokenisé : contrôles validés")


matrice X      : (240, 138) ( 240 documents x 138 mots )
densite        : 16.6% des cellules non nulles
somme ligne 0  : 1.0 (frequences relatives)

baseline majoritaire sur le test : 21.88%
vocabulaire alternatif, OOV et bigramme pré-tokenisé : contrôles validés


## 3. Le modèle : softmax, entropie croisée, gradient L2

Pour un document de caractéristiques `x` (V dimensions) et `K` catégories, le modèle calcule un score par catégorie, puis le normalise en probabilités :

$$z = xW + b, \qquad P(k \mid x) = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

La perte est l'entropie croisée moyenne, plus un terme L2 sur les poids :

$$L(W, b) = -\frac{1}{n}\sum_i \log P(y_i \mid x_i) + \frac{\lambda}{2}\lVert W \rVert^2$$

Le gradient a une forme remarquablement simple. En posant `dZ = (P - Y) / n` :

$$\frac{\partial L}{\partial W} = X^\top \, dZ + \lambda W, \qquad \frac{\partial L}{\partial b} = \sum_i dZ_i$$

Le biais `b` **n'est pas** régularisé : il ne multiplie aucune entrée, le pénaliser reviendrait à décourager les classes rares sans aucune raison.

Deux précautions d'implémentation, et elles ne sont pas cosmétiques :

- **stabilité numérique** : on retire le maximum de `z` avant l'exponentielle. La softmax est invariante à un décalage constant, donc le résultat est identique, mais l'exponentielle ne déborde plus ;
- **`log` écrêté** : `log(P)` utilise une probabilité bornée inférieurement à `1e-12`. Sans cela, une probabilité nulle sur la bonne classe envoie la perte à l'infini et la descente à `NaN`.

Notez ce que ce gradient dit : l'erreur `P - Y` est **la seule** information qui remonte. Là où Naive Bayes comptait des occurrences, ici chaque document pousse les poids dans la direction qui corrige sa propre probabilité.

In [4]:
def softmax(Z):
    """Softmax stable : retirer le maximum par ligne ne change pas le resultat."""
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


def perte_entropie_croisee(P, Y, W, lam):
    n = P.shape[0]
    donnees = -np.sum(Y * np.log(np.clip(P, 1e-12, 1.0))) / n
    return float(donnees + 0.5 * lam * np.sum(W * W))


def entrainer(X, Y, lam=1e-3, lr=0.5, iterations=600, graine=0):
    """Descente de gradient pleine passe. Rend (W, b, historique de perte)."""
    generateur = np.random.RandomState(graine)
    n, v = X.shape
    k = Y.shape[1]
    W = generateur.normal(0.0, 0.01, size=(v, k))
    b = np.zeros(k)
    historique = []
    for _ in range(iterations):
        P = softmax(X @ W + b)
        historique.append(perte_entropie_croisee(P, Y, W, lam))
        dZ = (P - Y) / n
        W -= lr * (X.T @ dZ + lam * W)
        b -= lr * dZ.sum(axis=0)
    historique.append(perte_entropie_croisee(softmax(X @ W + b), Y, W, lam))
    return W, b, np.array(historique)


def predire(X, W, b):
    """Etiquette predite pour chaque ligne de X."""
    return np.array([CATEGORIES[i] for i in softmax(X @ W + b).argmax(axis=1)])


print("softmax et gradient definis")
print("verification : softmax somme a 1 ->", np.allclose(softmax(np.array([[1.0, 2.0, 3.0]])).sum(), 1.0))
# Différences finies indépendantes sur poids et biais, loin de l'écrêtage.
x0 = np.array([[1., 2.], [-1., 0.5], [0.4, -0.2]])
y0 = np.array([[1., 0.], [0., 1.], [1., 0.]])
w0 = np.array([[0.1, -0.2], [0.3, 0.4]])
b0 = np.array([0.2, -0.1])
lam0, eps = 0.03, 1e-6
dz = (softmax(x0 @ w0 + b0) - y0) / len(x0)
gw, gb = x0.T @ dz + lam0 * w0, dz.sum(axis=0)
def perte_fixture(w, b):
    return perte_entropie_croisee(softmax(x0 @ w + b), y0, w, lam0)
for i, j in np.ndindex(w0.shape):
    delta = np.zeros_like(w0); delta[i, j] = eps
    assert np.isclose(gw[i, j], (perte_fixture(w0 + delta, b0) - perte_fixture(w0 - delta, b0)) / (2 * eps), atol=1e-7)
for j in range(len(b0)):
    delta = np.zeros_like(b0); delta[j] = eps
    assert np.isclose(gb[j], (perte_fixture(w0, b0 + delta) - perte_fixture(w0, b0 - delta)) / (2 * eps), atol=1e-7)
assert np.allclose(softmax(np.array([[1000., 1001.]])), softmax(np.array([[0., 1.]])))
print("différences finies W et b et stabilité softmax : validées")


softmax et gradient definis
verification : softmax somme a 1 -> True
différences finies W et b et stabilité softmax : validées


Le gradient et la stabilité numérique sont contrôlés sur une petite fixture. Entraînons maintenant le modèle sur le banc complet.

In [5]:
# --- Entrainement : la courbe de convergence est le premier resultat mesurable ---
ITERATIONS = 1200
W, b, historique = entrainer(X_train, Y_train, lam=1e-3, lr=0.5, iterations=ITERATIONS, graine=0)

print(" iterations   perte (entropie croisee + L2)")
print(" ----------   ------------------------------")
for it in [0, 25, 50, 100, 200, 400, 800, ITERATIONS]:
    print(f" {it:10d}   {historique[it]:.6f}")

print()
print("perte initiale :", f"{historique[0]:.6f}", " (log 4 =", f"{np.log(4):.6f}", ", donc depart neutre)")
print("perte finale   :", f"{historique[-1]:.6f}")
print("reduction      :", f"{historique[0] / historique[-1]:.2f}x")
print("stagnation sur les 100 dernieres iterations :",
      f"{abs(historique[-1] - historique[-101]):.2e}")

 iterations   perte (entropie croisee + L2)
 ----------   ------------------------------
          0   1.386421
         25   1.360364
         50   1.342207
        100   1.308587
        200   1.250791
        400   1.164967
        800   1.068218
       1200   1.023107

perte initiale : 1.386421  (log 4 = 1.386294 , donc depart neutre)
perte finale   : 1.023107
reduction      : 1.36x
stagnation sur les 100 dernieres iterations : 8.36e-03


### Lecture de la convergence

La perte passe de 1,386421 à 1,023107 en 1 200 itérations, soit une réduction de 1,36 fois ; la variation sur les 100 dernières est encore de 0,00836. Le départ est proche de `log 4` parce que les petits poids rendent les probabilités presque uniformes. La baisse mesurée dépend ici du pas et du budget : la convexité seule ne garantit pas une baisse à chaque itération avec un pas constant arbitraire. La perte résiduelle ou une précision d'apprentissage inférieure à 100 % ne suffisent pas à exclure le surapprentissage.

In [6]:
# --- Precision hors echantillon MESUREE (le resultat du carnet) ---
pred_train = predire(X_train, W, b)
pred_test = predire(X_test, W, b)

acc_train = float((pred_train == y_train).mean())
acc_test = float((pred_test == y_test).mean())
acc_base = baseline_majoritaire(y_train, y_test)

print(f"baseline majoritaire      : {acc_base:.2%}")
print(f"regression logistique train : {acc_train:.2%}")
print(f"regression logistique TEST  : {acc_test:.2%}   <-- hors echantillon")
print(f"gain sur la baseline        : {(acc_test - acc_base) * 100:+.2f} points de pourcentage")
print(f"ecart train - test          : {acc_train - acc_test:+.2%}")

# --- Matrice de confusion sur le jeu de test ---
mat = np.zeros((len(CATEGORIES), len(CATEGORIES)), dtype=int)
for vrai, pred in zip(y_test, pred_test):
    mat[CATEGORIES.index(vrai), CATEGORIES.index(pred)] += 1

entete = "vrai \\ pred".ljust(12) + "".join(c.rjust(10) for c in CATEGORIES)
print("\n" + entete)
print("-" * len(entete))
for i, c in enumerate(CATEGORIES):
    ligne = c.ljust(12) + "".join(str(mat[i, j]).rjust(10) for j in range(len(CATEGORIES)))
    print(ligne)

print("\nprecision par classe :")
for i, c in enumerate(CATEGORIES):
    support = mat[i].sum()
    bonne = mat[i, i]
    print(f"  {c:8} {bonne:3d}/{support:3d} = {bonne / support:.2%}" if support else f"  {c:8} aucun document")

baseline majoritaire      : 21.88%
regression logistique train : 86.11%
regression logistique TEST  : 75.00%   <-- hors echantillon
gain sur la baseline        : +53.12 points de pourcentage
ecart train - test          : +11.11%

vrai \ pred      sport      tech   science   finance
----------------------------------------------------
sport               18         2         2         0
tech                 0        20         2         0
science              0         1        19         1
finance              6         8         2        15

precision par classe :
  sport     18/ 22 = 81.82%
  tech      20/ 22 = 90.91%
  science   19/ 21 = 90.48%
  finance   15/ 31 = 48.39%


### Lecture du résultat

La précision de test atteint 75,00 % contre 21,88 % pour la baseline, soit un gain de 53,12 points ; l'entraînement atteint 86,11 %, un écart de 11,11 points. La matrice montre un rappel finance de 15/31 (48,39 %) : six documents sont classés sport et huit tech. La précision globale masque donc une difficulté de cette classe. Un seul découpage ne suffit pas à conclure que le modèle généralise ; il faut répéter l'évaluation sur d'autres splits ou en validation croisée.

### Exercice 1 — Mini-lots

Implémentez la descente par mini-lots : mélangez les indices à chaque époque, calculez `P - Y` sur le seul lot et mettez à jour poids et biais. Mesurez la perte atteinte par lots de 16 contre la descente pleine passe pour un même nombre de documents vus. Les deux fixtures inversent les étiquettes et vérifient que l'historique final correspond au modèle rendu.

In [7]:
# Exercice 1 — Descente de gradient stochastique (mini-lots)
#
# TODO etudiant : implementer entrainer_sgd ci-dessous.
# Le gradient d'un mini-lot est une estimation bruitee du gradient du corpus entier.
# Il est plus grossier par iteration, mais chaque iteration coute beaucoup moins cher.
#
# Indice 1 : melanger les indices a chaque epoque avec np.random.RandomState(graine).permutation.
# Indice 2 : le gradient se calcule sur le seul lot courant ; dZ = (P - Y_lot) / taille_du_lot.
# Indice 3 : garder les iterations a nombre constant de documents vus, pour comparer honnetement.
#
# Mesure attendue : la perte atteinte pour un meme nombre de documents vus, en mini-lots
#                  de 16, contre la descente pleine passe de la section 3.

def entrainer_sgd(X, Y, taille_lot=16, lam=1e-3, lr=0.5, epochs=50, graine=0):
    """Descente de gradient stochastique par mini-lots. Rend (W, b, historique de perte)."""
    # TODO etudiant
    return None


# Deux jeux séparables à étiquettes inversées : vérifier formes, finitude et progrès.
x_ex = np.array([[1., 0.], [0., 1.], [1., 0.], [0., 1.]])
y_ex = np.array([[1., 0.], [0., 1.], [1., 0.], [0., 1.]])
resultat = entrainer_sgd(x_ex, y_ex, taille_lot=2, epochs=50)
if resultat is None:
    print("Exercice 1 à compléter")
else:
    try:
        valide = True
        for cibles in (y_ex, y_ex[:, ::-1]):
            weights, bias, hist = entrainer_sgd(x_ex, cibles, taille_lot=2, epochs=50)
            valide &= (weights.shape == (2, 2) and bias.shape == (2,) and len(hist) >= 2
                       and np.isfinite(weights).all() and np.isfinite(bias).all()
                       and np.isfinite(hist).all() and hist[-1] < hist[0]
                       and np.isclose(hist[-1], perte_entropie_croisee(softmax(x_ex @ weights + bias), cibles, weights, 1e-3), atol=1e-8)
                       and np.array_equal((x_ex @ weights + bias).argmax(axis=1), cibles.argmax(axis=1)))
        print("Fixtures mini-lots validées" if valide else "Réponse incorrecte")
    except (TypeError, ValueError, IndexError):
        print("Réponse incorrecte")


Exercice 1 à compléter


## 4. Cross-check SOTA — la machine from scratch contre `scikit-learn`

`LogisticRegression` reçoit la même matrice et les mêmes étiquettes. La précision, l'accord par document, le cosinus et les normes des coefficients décrivent la ressemblance des résultats. Les régularisations et budgets diffèrent : ces mesures ne prouvent ni l'identité de l'objectif, ni celle de l'optimum. Le contrôle numérique indépendant du gradient, effectué ci-dessus, vérifie une propriété distincte de l'implémentation.

In [8]:
from sklearn.linear_model import LogisticRegression

reference = LogisticRegression(C=1e3, max_iter=5000)
reference.fit(X_train, y_train)
pred_sklearn = reference.predict(X_test)

acc_sklearn = float((pred_sklearn == y_test).mean())
accord = float((pred_sklearn == pred_test).mean())

# Les classes de sklearn sont triees : on remet les coefficients dans l'ordre de CATEGORIES.
ordre_classes = [list(reference.classes_).index(c) for c in CATEGORIES]
W_sklearn = reference.coef_[ordre_classes].T

cosinus = float((W * W_sklearn).sum() / (np.linalg.norm(W) * np.linalg.norm(W_sklearn)))

print("                            precision test")
print("---------------------------- --------------")
print(f"from scratch (ce carnet)   : {acc_test:.2%}")
print(f"scikit-learn (reference)   : {acc_sklearn:.2%}")
print(f"baseline majoritaire       : {acc_base:.2%}")
print()
print(f"accord des predictions, document par document : {accord:.2%} ({int(accord * len(y_test))}/{len(y_test)})")
print(f"similarite des poids appris (cosinus)         : {cosinus:.4f}")
print(f"norme des poids : from scratch {np.linalg.norm(W):.4f} | sklearn {np.linalg.norm(W_sklearn):.4f}")

                            precision test
---------------------------- --------------
from scratch (ce carnet)   : 75.00%
scikit-learn (reference)   : 75.00%
baseline majoritaire       : 21.88%

accord des predictions, document par document : 81.25% (78/96)
similarite des poids appris (cosinus)         : 0.7419
norme des poids : from scratch 13.8266 | sklearn 150.3282


### Lecture du cross-check

Sur ce découpage Python, les précisions de test valent 75,00 % chacune contre 21,88 % pour la baseline, mais les modèles ne s'accordent que sur 78 des 96 documents (81,25 %). Le cosinus des coefficients vaut 0,7419 et leurs normes 13,83 et 150,33 : les directions se ressemblent partiellement et leurs échelles diffèrent. Ces écarts reflètent notamment des régularisations et des budgets différents ; ils ne permettent pas d'attribuer une cause unique ni d'établir le même optimum.

### Exercice 2 — Banc déséquilibré

Construisez un banc déséquilibré à partir du générateur (par exemple 120 documents pour une classe et 20 pour chacune des autres), réentraînez le modèle puis calculez le **rappel** par classe : documents correctement prédits divisés par le support de chaque classe. Comparez ces rappels à la précision globale et expliquez ce qu'elle masque. Les fixtures contrôlent deux distributions de prédictions.

In [9]:
# Exercice 2 — Le banc desequilibre, et ce que la precision globale cache
#
# TODO etudiant :
#   1. construire un banc desequilibre (par exemple 120 documents d'une classe contre 20
#      par classe ailleurs) en reutilisant le generateur de la section 1 ;
#   2. reentrainer le modele sur ce banc ;
#   3. implementer precision_par_classe ci-dessous et l'appliquer.
#
# Indice : la matrice de confusion de la section 4 se generalise directement en
#          precision par classe ; inutile de reecrire un second calcul.
#
# Question a trancher a l'ecrit : sur ce banc, la precision globale suffit-elle a decrire
# le comportement du modele ? Repondre par la mesure, pas par l'intuition.

def precision_par_classe(y_vrai, y_pred, categories):
    """Rend un dict {categorie: precision}, la precision etant rappel = justes / support."""
    # TODO etudiant
    return None


resultat = precision_par_classe(["a", "a", "b"], ["a", "b", "b"], ["a", "b"])
if resultat is None:
    print("Exercice 2 à compléter")
else:
    try:
        autre = precision_par_classe(["a", "b", "b"], ["b", "a", "a"], ["a", "b"])
        valide = (set(resultat) == {"a", "b"} and set(autre) == {"a", "b"}
                  and all(np.isfinite(v) and 0 <= v <= 1 for v in list(resultat.values()) + list(autre.values()))
                  and np.isclose(resultat["a"], 0.5) and np.isclose(resultat["b"], 1.0)
                  and np.isclose(autre["a"], 0) and np.isclose(autre["b"], 0))
        print("Fixtures rappel validées" if valide else "Réponse incorrecte")
    except (TypeError, ValueError, KeyError):
        print("Réponse incorrecte")


Exercice 2 à compléter


## 5. Ce que la régularisation fait — et ce qu'elle ne fait pas

`lambda` fixe le prix des grands poids. La théorie annonce trois régimes :

- **`lambda = 0`** : aucun prix. Les poids peuvent devenir grands pour séparer des documents presque identiques, et le modèle colle alors au bruit de son jeu d'apprentissage ;
- **`lambda` trop grand** : le prix écrase tout. Les poids restent près de zéro, le modèle prédit presque uniformément et rejoint la baseline ;
- **entre les deux** : le compromis, qu'on repère en regardant l'**écart** entre précision d'apprentissage et précision de test, pas la seule précision de test.

Ces trois régimes sont un raisonnement, pas un résultat. Ce carnet ne les récite pas : il les **met à l'épreuve** sur son propre banc. La cellule suivante balaie `lambda` à réglages d'optimisation constants (`lr`, itérations, graine) — sans cette discipline, on ne saurait pas ce qui a produit la différence — et affiche, pour chaque valeur, la norme des poids, les deux précisions et leur écart.

Le résultat peut contredire l'attendu. C'est arrivé, et la lecture qui suit le dit tel quel.

In [10]:
# --- Balayage de lambda : seule la regularisation varie ---
LAMBDAS = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

print("   lambda   perte finale   ||W||    acc_train   acc_test   ecart")
print("   ------   ------------   -----    ---------   --------   -----")
resultats = []
for lam in LAMBDAS:
    Wl, bl, hl = entrainer(X_train, Y_train, lam=lam, lr=0.5, iterations=ITERATIONS, graine=0)
    a_train = float((predire(X_train, Wl, bl) == y_train).mean())
    a_test = float((predire(X_test, Wl, bl) == y_test).mean())
    norme = float(np.linalg.norm(Wl))
    resultats.append((lam, hl[-1], norme, a_train, a_test))
    print(f"   {lam:6.4f}   {hl[-1]:12.6f}   {norme:6.3f}   {a_train:9.2%}   {a_test:8.2%}   {a_train - a_test:+.2%}")

meilleur = max(resultats, key=lambda r: r[4])
print(f"\nmeilleur lambda sur la precision de test : {meilleur[0]:.4f} (test {meilleur[4]:.2%})")
print(f"meilleur lambda sur l'ecart train-test minimal : "
      f"{min(resultats, key=lambda r: r[3] - r[4])[0]:.4f}")
print("\nrappel : la baseline majoritaire vaut", f"{acc_base:.2%}")

   lambda   perte finale   ||W||    acc_train   acc_test   ecart
   ------   ------------   -----    ---------   --------   -----
   0.0000       0.828751   17.892      86.11%     76.04%   +10.07%


   0.0001       0.854803   17.412      86.11%     76.04%   +10.07%


   0.0010       1.023107   13.827      86.11%     75.00%   +11.11%
   0.0100       1.308565    3.586      71.53%     60.42%   +11.11%


   0.1000       1.371866    0.392      50.69%     34.38%   +16.32%


   1.0000       1.378842    0.040      27.08%     21.88%   +5.21%

meilleur lambda sur la precision de test : 0.0000 (test 76.04%)
meilleur lambda sur l'ecart train-test minimal : 1.0000

rappel : la baseline majoritaire vaut 21.88%


### Lecture du balayage

Dans cette exécution, la norme des poids et la précision de test diminuent lorsque `lambda` augmente ; les sorties ci-dessus donnent les valeurs et l'écart train-test. Il s'agit d'un seul découpage et d'un budget fini, pas d'une loi de monotonie. Une précision d'entraînement inférieure à 100 % ne prouve pas l'absence de surapprentissage, et le rapport de 138 caractéristiques à 144 documents ne suffit pas à prévoir l'effet de L2 : cette pénalisation peut aussi aider lorsque le nombre de documents dépasse celui des caractéristiques. Des bigrammes changeraient la représentation, mais leur effet reste à mesurer. Pour sélectionner `lambda`, utiliser une validation séparée ou une validation croisée, puis réserver le test à l'évaluation finale.

### Exercice 3 — Bigrammes et régularisation

Ajoutez à chaque liste d'unigrammes les paires consécutives, séparées par `_`. Construisez le vocabulaire sur ces jetons et leur matrice en fréquences relatives avec le chemin pré-tokenisé (`tokenizer=lambda tokens: tokens` en Python ; `ConstruireMatriceTokens` en C#), sans retokeniser les bigrammes. Réentraînez puis balayez `lambda` aux mêmes réglages ; comparez la forme obtenue au tableau précédent sans supposer d'avance l'effet de L2.

In [11]:
# Exercice 3 — Bigrammes, et le retour de la regularisation
#
# La section 5 mesure le balayage sur ce banc. Les bigrammes modifient la
# representation ; leur effet sur la regularisation reste a mesurer.
#
# TODO etudiant :
#   1. implementer bigrammes(documents) ci-dessous : rendre, pour chaque document, la liste
#      de ses mots SUIVIS de ses paires de mots consecutifs (par exemple "le_match") ;
#   2. construire la matrice avec construire_matrice(jetons, vocab_bi, tokenizer=lambda tokens: tokens) ;
#   3. reentrainer et rebalayer lambda (0, 1e-4, 1e-3, 1e-2, 1e-1).
#
# Indice 1 : le vocabulaire passe de ~138 a plusieurs milliers d'entrees, donc
#            caracteristiques >> documents. Ce rapport ne suffit pas a predire l'effet de L2.
# Indice 2 : la normalisation en frequences relatives reste valable telle quelle.
# Indice 3 : comparer la FORME du balayage a celle de la section 5, pas seulement le
#            meilleur chiffre — c'est la question posee.

def bigrammes(docs):
    """Rend une liste de listes de tokens : les mots, puis les paires consecutives."""
    # TODO etudiant
    return None


jetons = bigrammes(["le match match", "but", ""])
if jetons is None:
    print("Exercice 3 à compléter")
else:
    print("Fixture bigrammes validée" if jetons == [["le", "match", "match", "le_match", "match_match"], ["but"], []] else "Réponse incorrecte")

    if jetons == [["le", "match", "match", "le_match", "match_match"], ["but"], []]:
        jetons_banc = bigrammes(documents)
        vocab_bi = sorted({m for doc in jetons_banc for m in doc})
        print(f"vocabulaire unigrammes : {len(vocabulaire)} | unigrammes et bigrammes : {len(vocab_bi)}")
        print(f"rapport caractéristiques / documents train : {len(vocab_bi) / len(idx_train):.1f} pour 1")


Exercice 3 à compléter


## 6. Synthèse

Le carnet construit un banc de 240 documents chevauchants, représente les textes par fréquences relatives et entraîne une softmax avec gradient et pénalité L2. Les différences finies contrôlent les poids et biais ; la matrice de confusion complète la précision hors échantillon.

Sur ce découpage Python, la perte passe de 1,386421 à 1,023107 et la précision test vaut 75,00 % contre 21,88 % pour la baseline. Le témoin scikit-learn atteint aussi 75,00 %, mais ne s'accorde que sur 78/96 documents. Les budgets et régularisations diffèrent : ces mesures ne démontrent pas le même optimum.

Le balayage décrit une baisse de la norme de 17,892 à 0,040 et de la précision test de 76,04 % à 21,88 % entre les valeurs extrêmes de `lambda`. Ce résultat concerne un split et un budget finis ; sélectionner `lambda` demande une validation distincte. Comparer Naive Bayes et régression logistique sur ce banc à représentation identique reste une expérience à réaliser.

## 7. Exercices

Les trois exercices sont répartis dans le carnet : [mini-lots](#exercice-1), [rappel par classe](#exercice-2) et [bigrammes](#exercice-3). Ils prolongent l'optimisation, l'évaluation d'un banc déséquilibré et l'étude de la régularisation.

## Références

- D. Jurafsky et J. H. Martin, *Speech and Language Processing*, 3e édition (draft), chapitre « Logistic Regression » — la dérivation de la softmax, de l'entropie croisée et de son gradient y est traitée sous l'angle du TAL.
- C. D. Manning, P. Raghavan et H. Schütze, *Introduction to Information Retrieval*, Cambridge University Press, 2008, chapitre 13 « Text classification and Naive Bayes » — le cadre génératif/discriminatif dont ce carnet et le carnet 10 occupent les deux pôles.
- `scikit-learn`, documentation de `sklearn.linear_model.LogisticRegression` — conventions de régularisation (`C`, multi-classe), utilisées ici comme juge extérieur.

## Navigation

- [<< Précédent — 10. Classification de texte Naive Bayes from scratch](10-Classement-Texte-NaiveBayes-FromScratch.ipynb) — le pôle génératif du duo, dont la section 1 de ce carnet reprend le corpus comme socle de vocabulaire.
- [Index de la série](README.md) — le même point d'entrée que les dix autres carnets de la série.
- Carnet suivant : **aucun à ce jour** — ce carnet ferme la série ; le prochain numéro libre sera fixé à la consolidation de la série.

## À retenir

- La softmax calcule directement les probabilités de classe ; son gradient dépend de `P - Y`.
- La perte débute près de `log 4` et sa décroissance observée dépend du pas choisi.
- Sur ce découpage, la précision de test est de 75,00 % contre 21,88 % pour la baseline ; la matrice de confusion détaille les classes.
- Le témoin scikit-learn obtient ici 75,00 % et 81,25 % d'accord : même précision ne signifie pas mêmes prédictions. Des régularisations et budgets différents ne permettent pas de conclure à un même optimum.
- Le balayage est descriptif. L2 peut aider ou nuire selon les données et l'optimisation, y compris lorsque les documents sont plus nombreux que les caractéristiques.

## Vérifiez votre compréhension

1. Pourquoi la perte initiale est-elle proche de `log 4` ?
2. Que montre l'accord de 81,25 % quand les précisions de test sont toutes deux de 75 % ?
3. Pourquoi la comparaison des coefficients ne prouve-t-elle pas l'identité des optima ?
4. Pourquoi le nombre de caractéristiques ne suffit-il pas à prévoir l'effet de L2 ?

<details><summary>Réponses</summary>

1. Les petits poids initiaux produisent des probabilités presque uniformes sur quatre classes.
2. Les modèles se trompent sur des documents différents ; ils divergent sur 18 des 96 exemples.
3. Les budgets et régularisations diffèrent ; un cosinus et deux normes ne déterminent pas l'objectif optimisé.
4. L'effet dépend également du bruit, des données et de l'optimisation ; il doit être mesuré sur une validation indépendante.

</details>

## Pour aller plus loin

**Carnet précédent** : [10-Classement-Texte-NaiveBayes-FromScratch.ipynb](10-Classement-Texte-NaiveBayes-FromScratch.ipynb) — le pôle génératif du même problème, dont ce carnet réutilise le corpus comme socle de vocabulaire. La comparaison génératif/discriminatif sur **ce** banc chevauchant, à représentation identique, est l'expérience naturelle que la synthèse désigne — et qu'elle laisse ouverte.

À ce jour, ce carnet ferme la série NLP.

- **Lectures** : Ng & Jordan, « On Discriminative vs. Generative Classifiers : Logistic Regression and Naive Bayes » (NIPS 2001) — le couple exact que les carnets 10 et 11 incarnent, et pourquoi le génératif devance le discriminatif sur les petits échantillons ; Jurafsky & Martin, *Speech and Language Processing* (3e éd., draft), ch. « Logistic Regression ».
- **Dans le dépôt** : [02_NGrammes_Modeles_De_Langue.ipynb](02_NGrammes_Modeles_De_Langue.ipynb) — l'autre tradition d'estimation de la série ; la série `RAG-et-Memoire-Semantique`, où la représentation en sac-de-mots devient une question de recherche d'information.